In [1]:
pip install openpyxl -qq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import pandas as pd

DATA_DIR = Path(os.getenv('PROJECT_DIR')) / 'paper'
BOOLEAN_COLUMNS = [
    'Irrelevant title/abstract', 'Does not regard DAIC-WOZ', 'Does not use MAE',
    'Does not introduce new model', 'Not in English', 'Not retrieved',
    'Discarded after full-text review', 'Included'
]

# load data
screening_df = pd.read_excel(DATA_DIR / 'screening.xlsx', keep_default_na=False)
screening_continued_df = pd.read_excel(DATA_DIR / 'screening_continued.xlsx', keep_default_na=False)

# remove bottom line rows
screening_df = screening_df.iloc[:screening_df[screening_df['Title'] == 'Total'].index[0]]
screening_continued_df = screening_continued_df.iloc[:screening_continued_df[screening_continued_df['Title'] == 'Total'].index[0]]

# combine dataframes into single one
df = pd.concat([screening_df, screening_continued_df])
df.rename({'Data partitioning': 'Data proportions'}, axis=1, inplace=True)

# cast some values to boolean
for column in BOOLEAN_COLUMNS:
    df[column] = df[column].apply(lambda x: bool(x) if x in [0, 1] else x)

# preserve only entries which underwent quality screening
df = df[(df['Irrelevant title/abstract'] == False) & (df['Discarded after full-text review'] == False)]

# drop columns irrelevant for this analysis
df.drop(
    [
        'Title', 'Link', 'Source', 'Abstract', 'Irrelevant title/abstract',
        'Dataset (if not DAIC-WOZ)', 'Does not regard DAIC-WOZ',
        'Does not use MAE', 'Does not introduce new model', 'Not in English',
        'Not retrieved', 'Discarded after full-text review', '-', '--',
        # 'Comment'
    ],
    axis=1,
    inplace=True
)

df.columns

Index(['Data preprocessing', 'Data proportions', 'Partitions disjoint level',
       'Model description', 'Training details', 'Final model selection',
       'Data modality', 'Dev. MAE', 'Dev. RMSE', 'Dev. R2', 'Test. MAE',
       'Test. RMSE', 'Test. R2', 'Code available', 'Repeated experiments',
       'Uses interviewer's prompts', 'Participant inter-partition leakage',
       'Included', 'Comment'],
      dtype='object')

### Reporting issues

In [3]:
def print_issue(count, issue_name):
    print(f'{issue_name} {100*count/len(df):.1f}% ({count})')

In [4]:
quality_criteria = [
    'Data preprocessing', 'Data proportions', 'Partitions disjoint level',
    'Model description', 'Training details', 'Final model selection'
]
for column in quality_criteria:
    count = df[column].value_counts()['-']
    issue_name = column + ':' + ' ' * (25 - len(column)) 
    print_issue(count=count, issue_name=issue_name)

# data partitioning
count = ((df['Data proportions'] == '-') | (df['Partitions disjoint level'] == '-')).sum()
issue_name = 'Overall data partitioning' + ':' + ' ' * (25 - len('Overall data partitioning'))
print_issue(count=count, issue_name=issue_name)

Data preprocessing:        34.8% (23)
Data proportions:          19.7% (13)
Partitions disjoint level: 27.3% (18)
Model description:         27.3% (18)
Training details:          71.2% (47)
Final model selection:     60.6% (40)
Overall data partitioning: 37.9% (25)


In [5]:
no_results_mask = df[['Dev. MAE', 'Dev. RMSE', 'Dev. R2', 'Test. MAE']].apply(
    lambda row: row.values.tolist() == ['-'] * 4,
    axis=1
)
unknown_partition_mask = df[no_results_mask]['Comment'].apply(
    lambda x: 'unknown partition' in x.lower()
)
unknown_partition_count = unknown_partition_mask.sum()
print_issue(count=unknown_partition_count, issue_name='Results reported for an unknown partition:')

Results reported for an unknown partition: 12.1% (8)


In [6]:
results_inconsistencies_mask = df[no_results_mask][~unknown_partition_mask]['Comment'].apply(
    lambda x: 'inconsist' in x.lower()
)
results_inconsistencies_count = results_inconsistencies_mask.sum()
print_issue(count=results_inconsistencies_count, issue_name='Inconsistencies in reported results:')

Inconsistencies in reported results: 3.0% (2)


---

In [7]:
boolean_criteria = df[quality_criteria].map(lambda x: x == '-')
boolean_criteria['Data partitioning'] = boolean_criteria.apply(
    lambda row: row['Data proportions'] or row['Partitions disjoint level'],
    axis=1
)
boolean_criteria.drop(['Partitions disjoint level', 'Data proportions'], axis=1, inplace=True)
criteria_compliance = (5 - boolean_criteria.sum(axis=1)).value_counts().sort_index()

In [8]:
# number of studies compliant with less than `index + 1` number of criteria
for num_criteria, count in criteria_compliance.sort_index(ascending=True).cumsum().items():
    if num_criteria == 0:
        print(f'Studies not compliant with any criterion:   {100*count/len(df):.1f}% ({count})')
    elif num_criteria == 1:
        print(f'Studies compliant with 1 criterion or less: {100*count/len(df):.1f}% ({count})')
    elif num_criteria == 5:
        pass
    else:
        print(f'Studies compliant with {num_criteria} criteria or less:  {100*count/len(df):.1f}% ({count})')

Studies not compliant with any criterion:   9.1% (6)
Studies compliant with 1 criterion or less: 16.7% (11)
Studies compliant with 2 criteria or less:  42.4% (28)
Studies compliant with 3 criteria or less:  71.2% (47)
Studies compliant with 4 criteria or less:  92.4% (61)


In [9]:
# number of studies compliant with at least `index` number of criteria
for num_criteria, count in criteria_compliance.sort_index(ascending=False).cumsum().sort_index().items():
    if num_criteria == 0:
        pass
    elif num_criteria == 1:
        print(f'Studies compliant with at least 1 criterion: {100*count/len(df):.1f}% ({count})')
    elif num_criteria == 5:
        print(f'Studies compliant with all 5 criteria:       {100*count/len(df):.1f}% ({count})')
    else:
        print(f'Studies compliant with at least {num_criteria} criteria:  {100*count/len(df):.1f}% ({count})')

Studies compliant with at least 1 criterion: 90.9% (60)
Studies compliant with at least 2 criteria:  83.3% (55)
Studies compliant with at least 3 criteria:  57.6% (38)
Studies compliant with at least 4 criteria:  28.8% (19)
Studies compliant with all 5 criteria:       7.6% (5)


### Methodological flaws

In [10]:
code_available_count = df['Code available'].value_counts()['Yes']
print_issue(count=code_available_count, issue_name='Studies with publicly available code:')

Studies with publicly available code: 15.2% (10)


In [11]:
repeated_experiments_count = df['Repeated experiments'].value_counts()['Yes']
print_issue(count=repeated_experiments_count, issue_name='Studies with repeated experiments:')

Studies with repeated experiments: 18.2% (12)


In [12]:
use_interviewer_count = df['Uses interviewer\'s prompts'].value_counts()['Yes']
use_interviewer_missing_count = df['Uses interviewer\'s prompts'].value_counts()['-']
print_issue(count=use_interviewer_count, issue_name='Studies which use interviewer\'s turns/utterances:')
print_issue(count=use_interviewer_missing_count, issue_name='                                  (missing data):')

Studies which use interviewer's turns/utterances: 42.4% (28)
                                  (missing data): 25.8% (17)


In [13]:
subject_leakage_count = df['Participant inter-partition leakage'].value_counts()['Yes']
subject_leakage_missing_count = df['Participant inter-partition leakage'].value_counts()['-']
print_issue(count=subject_leakage_count, issue_name='Studies with subject leakage:')
print_issue(count=subject_leakage_missing_count, issue_name='              (missing data):')

Studies with subject leakage: 9.1% (6)
              (missing data): 24.2% (16)


In [14]:
report_r2_mask = (df['Dev. R2'] != '-') | (df['Test. R2'] != '-')
report_r2_count = report_r2_mask.sum()
print_issue(count=report_r2_count, issue_name='Studies reporting R2:')

Studies reporting R2: 6.1% (4)


In [15]:
held_out_test_set_mask = (df['Test. MAE'] != '-') & (df['Participant inter-partition leakage'] == 'No')
held_out_test_set_count = held_out_test_set_mask.sum()
print_issue(count=held_out_test_set_count, issue_name='Studies reporting results for held-out test set:')

Studies reporting results for held-out test set: 30.3% (20)
